In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import gc
import json
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cv2
import nibabel as nib
import tensorflow as tf

from scipy.ndimage import binary_fill_holes
from scipy.spatial.distance import cdist


SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


print("TensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPU(s):", gpus)

TensorFlow: 2.20.0
GPU(s): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/CAMUS_FINAL"
)

PHASE1F_DIR = (
    PROJECT_DIR /
    "phase1_segmentation" /
    "phase1F_final_model"
)

PHASE2_DIR = (
    PROJECT_DIR /
    "phase2_quantitative"
)

P2A_DIR = (
    PHASE2_DIR /
    "phase2A_measurement_method"
)

P2A_CONFIG_DIR = (
    P2A_DIR /
    "configs"
)

P2A_TABLE_DIR = (
    P2A_DIR /
    "tables"
)

P2A_PLOT_DIR = (
    P2A_DIR /
    "plots"
)

P2A_PREDICTION_DIR = (
    P2A_DIR /
    "predictions"
)


for folder in [
    P2A_CONFIG_DIR,
    P2A_TABLE_DIR,
    P2A_PLOT_DIR,
    P2A_PREDICTION_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


print("Phase 2A directory:")
print(P2A_DIR)

print("\n✓ Phase 2A folders ready.")

Phase 2A directory:
/content/drive/MyDrive/CAMUS_FINAL/phase2_quantitative/phase2A_measurement_method

✓ Phase 2A folders ready.


In [ ]:
MODEL_POINTER_PATH = (
    PHASE1F_DIR /
    "configs" /
    "phase2_segmentation_model_pointer.json"
)


assert MODEL_POINTER_PATH.exists()


with open(
    MODEL_POINTER_PATH,
    "r"
) as f:
    model_pointer = json.load(f)


FINAL_MODEL_PATH = Path(
    model_pointer[
        "model_path"
    ]
)

FINAL_CONFIG_PATH = Path(
    model_pointer[
        "config_path"
    ]
)

FINAL_MANIFEST_PATH = Path(
    model_pointer[
        "manifest_path"
    ]
)

EXPECTED_MODEL_SHA256 = (
    model_pointer[
        "sha256"
    ]
)


assert FINAL_MODEL_PATH.exists()
assert FINAL_CONFIG_PATH.exists()
assert FINAL_MANIFEST_PATH.exists()

assert (
    model_pointer[
        "frozen"
    ]
    is True
)


print("=" * 70)
print("FROZEN PHASE 1 MODEL LOCATED")
print("=" * 70)

print("\nModel:")
print(FINAL_MODEL_PATH)

print("\nConfig:")
print(FINAL_CONFIG_PATH)

print("\nManifest:")
print(FINAL_MANIFEST_PATH)

print(
    "\n✓ Phase 2 is linked to "
    "the frozen Phase 1F model."
)

FROZEN PHASE 1 MODEL LOCATED

Model:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1F_final_model/models/final_lv_segmentation_model.keras

Config:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1F_final_model/configs/final_segmentation_config.json

Manifest:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1F_final_model/configs/final_segmentation_manifest.json

✓ Phase 2 is linked to the frozen Phase 1F model.


In [ ]:
def sha256_file(file_path):

    sha = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            sha.update(chunk)

    return sha.hexdigest()


CURRENT_MODEL_SHA256 = (
    sha256_file(
        FINAL_MODEL_PATH
    )
)


assert (
    CURRENT_MODEL_SHA256
    ==
    EXPECTED_MODEL_SHA256
)


print("=" * 70)
print("MODEL INTEGRITY VERIFIED")
print("=" * 70)

print(
    "\nSHA-256 matches Phase 1F manifest:"
)

print(True)

print(
    "\n✓ Correct frozen segmentation "
    "model confirmed."
)

MODEL INTEGRITY VERIFIED

SHA-256 matches Phase 1F manifest:
True

✓ Correct frozen segmentation model confirmed.


In [ ]:
with open(
    FINAL_CONFIG_PATH,
    "r"
) as f:
    segmentation_config = json.load(f)


assert (
    segmentation_config[
        "frozen"
    ]
    is True
)

assert (
    segmentation_config[
        "input"
    ][
        "resolution"
    ]
    == [256, 256]
)

assert np.isclose(
    segmentation_config[
        "inference"
    ][
        "threshold"
    ],
    0.50
)


SEGMENTATION_RESOLUTION = 256
SEGMENTATION_THRESHOLD = 0.50


print("=" * 70)
print("SEGMENTATION CONFIGURATION")
print("=" * 70)

print("\nResolution : 256×256")
print("Threshold  : 0.50")
print(
    "Postprocess: largest connected "
    "component + hole filling"
)

print(
    "\n✓ Segmentation configuration locked."
)

SEGMENTATION CONFIGURATION

Resolution : 256×256
Threshold  : 0.50
Postprocess: largest connected component + hole filling

✓ Segmentation configuration locked.


In [ ]:
import zipfile

OUTER_ZIP_PATH = Path(
    "/content/drive/MyDrive/download.zip"
)

OUTER_EXTRACT_DIR = Path(
    "/content/camus_data"
)

INNER_ARCHIVE_PATH = (
    OUTER_EXTRACT_DIR /
    "download"
)

INNER_EXTRACT_DIR = Path(
    "/content/camus_data_final"
)

DATASET_PATH = (
    INNER_EXTRACT_DIR /
    "database_nifti"
)


if not DATASET_PATH.exists():

    assert OUTER_ZIP_PATH.exists(), (
        f"Archive not found: {OUTER_ZIP_PATH}"
    )

    if not INNER_ARCHIVE_PATH.exists():

        print("Extracting outer archive...")

        OUTER_EXTRACT_DIR.mkdir(
            parents=True,
            exist_ok=True
        )

        with zipfile.ZipFile(
            OUTER_ZIP_PATH,
            "r"
        ) as archive:

            archive.extractall(
                OUTER_EXTRACT_DIR
            )

        print("✓ Outer archive extracted.")


    print("Extracting CAMUS archive...")

    INNER_EXTRACT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        INNER_ARCHIVE_PATH,
        "r"
    ) as archive:

        archive.extractall(
            INNER_EXTRACT_DIR
        )


assert DATASET_PATH.exists()

print("\n✓ CAMUS dataset ready:")
print(DATASET_PATH)

Extracting outer archive...
✓ Outer archive extracted.
Extracting CAMUS archive...

✓ CAMUS dataset ready:
/content/camus_data_final/database_nifti


In [ ]:
SPLIT_PATH = (
    PROJECT_DIR /
    "splits" /
    "patient_split_seed42.json"
)


with open(
    SPLIT_PATH,
    "r"
) as f:
    patient_split = json.load(f)


train_ids = patient_split.get(
    "train_ids",
    patient_split.get("train")
)

val_ids = patient_split.get(
    "val_ids",
    patient_split.get(
        "validation",
        patient_split.get("val")
    )
)

test_ids = patient_split.get(
    "test_ids",
    patient_split.get("test")
)


assert len(train_ids) == 400
assert len(val_ids) == 50
assert len(test_ids) == 50


print("Train patients :", len(train_ids))
print("Val patients   :", len(val_ids))
print("Test patients  :", len(test_ids))

print(
    "\nPhase 2 final evaluation patients:",
    len(test_ids)
)

Train patients : 400
Val patients   : 50
Test patients  : 50

Phase 2 final evaluation patients: 50


In [ ]:
def parse_info_cfg(cfg_path):

    info = {}

    with open(
        cfg_path,
        "r",
        encoding="utf-8"
    ) as f:

        for raw in f:

            line = raw.strip()

            if (
                not line
                or line.startswith("#")
                or ":" not in line
            ):
                continue

            key, value = line.split(
                ":",
                1
            )

            key = key.strip()
            value = value.strip()

            try:
                value = int(value)

            except ValueError:

                try:
                    value = float(value)

                except ValueError:
                    pass

            info[key] = value

    return info


def load_nifti_2d(file_path):

    nii = nib.load(
        str(file_path)
    )

    data = np.asarray(
        nii.get_fdata()
    )

    if (
        data.ndim == 3
        and data.shape[-1] == 1
    ):
        data = data[:, :, 0]

    if data.ndim != 2:

        raise ValueError(
            f"Expected 2D NIfTI, "
            f"got {data.shape}"
        )

    spacing = tuple(
        float(x)
        for x in
        nii.header.get_zooms()[:2]
    )

    return (
        data,
        spacing
    )


def load_patient_data(
    patient_id
):

    patient_dir = (
        DATASET_PATH /
        patient_id
    )

    clinical = parse_info_cfg(
        patient_dir /
        "Info_4CH.cfg"
    )

    patient = {

        "patient_id":
            patient_id,

        "clinical":
            clinical
    }


    for phase in [
        "ED",
        "ES"
    ]:

        image, spacing = (
            load_nifti_2d(
                patient_dir /
                f"{patient_id}_4CH_{phase}.nii.gz"
            )
        )

        gt, gt_spacing = (
            load_nifti_2d(
                patient_dir /
                f"{patient_id}_4CH_{phase}_gt.nii.gz"
            )
        )


        assert np.allclose(
            spacing,
            gt_spacing
        )


        patient[phase] = {

            "image":
                image.astype(
                    np.float32
                ),

            "mask":
                (gt == 1),

            "spacing":
                spacing,

            "original_shape":
                image.shape
        }


    return patient


sample_patient = load_patient_data(
    test_ids[0]
)


print("=" * 70)
print("SAMPLE PHYSICAL DATA CHECK")
print("=" * 70)

print(
    "\nPatient:",
    sample_patient[
        "patient_id"
    ]
)

for phase in ["ED", "ES"]:

    print(
        f"\n{phase}:"
    )

    print(
        "Shape:",
        sample_patient[
            phase
        ][
            "original_shape"
        ]
    )

    print(
        "Spacing:",
        sample_patient[
            phase
        ][
            "spacing"
        ]
    )

    print(
        "LV pixels:",
        int(
            sample_patient[
                phase
            ][
                "mask"
            ].sum()
        )
    )

SAMPLE PHYSICAL DATA CHECK

Patient: patient0173

ED:
Shape: (591, 487)
Spacing: (0.30799999833106995, 0.30799999833106995)
LV pixels: 30144

ES:
Shape: (591, 487)
Spacing: (0.30799999833106995, 0.30799999833106995)
LV pixels: 17039


In [ ]:
def preprocess_for_segmentation(
    image,
    target_size=256
):

    image = np.asarray(
        image,
        dtype=np.float32
    )

    original_h, original_w = (
        image.shape
    )


    resized = cv2.resize(
        image,
        (
            target_size,
            target_size
        ),
        interpolation=cv2.INTER_LINEAR
    )


    minimum = float(
        resized.min()
    )

    maximum = float(
        resized.max()
    )


    if maximum > minimum:

        normalized = (
            resized - minimum
        ) / (
            maximum - minimum
        )

    else:

        normalized = (
            np.zeros_like(
                resized
            )
        )


    scale_x = (
        original_w /
        target_size
    )

    scale_y = (
        original_h /
        target_size
    )


    return (
        normalized.astype(
            np.float32
        ),
        (
            scale_x,
            scale_y
        )
    )


def preprocess_mask(
    mask,
    target_size=256
):

    resized = cv2.resize(
        np.asarray(
            mask
        ).astype(
            np.uint8
        ),
        (
            target_size,
            target_size
        ),
        interpolation=cv2.INTER_NEAREST
    )

    return (
        resized > 0
    )


def effective_spacing(
    native_spacing,
    scale_factors
):

    spacing_x, spacing_y = map(
        float,
        native_spacing
    )

    scale_x, scale_y = map(
        float,
        scale_factors
    )

    return (
        spacing_x * scale_x,
        spacing_y * scale_y
    )


print(
    "✓ Resize geometry and "
    "effective-spacing functions ready."
)

✓ Resize geometry and effective-spacing functions ready.


In [ ]:
sample_phase = "ED"

sample_image = (
    sample_patient[
        sample_phase
    ][
        "image"
    ]
)

sample_mask_native = (
    sample_patient[
        sample_phase
    ][
        "mask"
    ]
)

sample_native_spacing = (
    sample_patient[
        sample_phase
    ][
        "spacing"
    ]
)


_, sample_scale = (
    preprocess_for_segmentation(
        sample_image
    )
)

sample_mask_resized = (
    preprocess_mask(
        sample_mask_native
    )
)

sample_effective_spacing = (
    effective_spacing(
        sample_native_spacing,
        sample_scale
    )
)


native_area_mm2 = (

    sample_mask_native.sum()

    *

    sample_native_spacing[0]

    *

    sample_native_spacing[1]
)


resized_area_mm2 = (

    sample_mask_resized.sum()

    *

    sample_effective_spacing[0]

    *

    sample_effective_spacing[1]
)


area_difference_pct = (

    (
        resized_area_mm2
        -
        native_area_mm2
    )

    /

    native_area_mm2

    *

    100.0
)


print("=" * 70)
print("PHYSICAL AREA SANITY CHECK")
print("=" * 70)

print(
    f"\nNative area  : "
    f"{native_area_mm2:.2f} mm²"
)

print(
    f"Resized area : "
    f"{resized_area_mm2:.2f} mm²"
)

print(
    f"Difference   : "
    f"{area_difference_pct:+.3f}%"
)

print(
    "\nNative spacing:",
    sample_native_spacing
)

print(
    "Effective resized spacing:",
    sample_effective_spacing
)

PHYSICAL AREA SANITY CHECK

Native area  : 2859.58 mm²
Resized area : 2855.50 mm²
Difference   : -0.143%

Native spacing: (0.30799999833106995, 0.30799999833106995)
Effective resized spacing: (0.5859218718251213, 0.7110468711471185)


In [ ]:
phase2a_protocol = {

    "phase":
        "Phase 2A — LV Quantitative Measurement Method",

    "purpose":
        (
            "Establish and validate the quantitative "
            "measurement methodology used downstream "
            "of the frozen LV segmentation model."
        ),

    "segmentation_model_frozen":
        True,

    "segmentation_model":
        str(
            FINAL_MODEL_PATH
        ),

    "segmentation_model_sha256":
        EXPECTED_MODEL_SHA256,

    "segmentation_changes_allowed":
        False,

    "target_view":
        "A4C",

    "target_structure":
        "LV cavity",

    "phases":
        ["ED", "ES"],

    "physical_units": {
        "area":
            "mm^2",
        "length":
            "mm",
        "volume":
            "mL",
        "EF":
            "%"
    },

    "candidate_long_axis_methods": [
        "maximum_feret",
        "PCA_major_axis"
    ],

    "volume_method":
        "single-plane area-length",

    "volume_equation":
        "V = (8 / (3*pi)) * (A^2 / L)",

    "method_selection_principle":
        (
            "Geometry methods will first be evaluated "
            "using ground-truth LV masks so that "
            "measurement-model error is separated from "
            "segmentation error."
        ),

    "downstream_quantities": [
        "ED area",
        "ES area",
        "ED long-axis length",
        "ES long-axis length",
        "EDV",
        "ESV",
        "SV",
        "EF"
    ],

    "test_exposure_note":
        (
            "The current 50-patient evaluation partition "
            "was previously inspected during historical "
            "pilot development. It was excluded from the "
            "clean Phase 1A-1D model-selection process, "
            "but is not claimed to be a pristine external "
            "test set."
        )
}


P2A_PROTOCOL_PATH = (
    P2A_CONFIG_DIR /
    "phase2A_measurement_protocol.json"
)


with open(
    P2A_PROTOCOL_PATH,
    "w"
) as f:

    json.dump(
        phase2a_protocol,
        f,
        indent=2
    )


print("=" * 70)
print("PHASE 2A PROTOCOL FROZEN")
print("=" * 70)

print(
    "\nSegmentation model changes: NO"
)

print(
    "Candidate geometry methods:"
)

print(
    "  1. Maximum Feret"
)

print(
    "  2. PCA major axis"
)

print(
    "\nVolume method:"
)

print(
    "  Single-plane area-length"
)

print(
    "\nGround-truth geometry will "
    "be examined first."
)

print("\nSaved:")
print(P2A_PROTOCOL_PATH)

PHASE 2A PROTOCOL FROZEN

Segmentation model changes: NO
Candidate geometry methods:
  1. Maximum Feret
  2. PCA major axis

Volume method:
  Single-plane area-length

Ground-truth geometry will be examined first.

Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase2_quantitative/phase2A_measurement_method/configs/phase2A_measurement_protocol.json


In [ ]:
# ================================================================
# PHASE 2A — CELL 13
# INSPECT AVAILABLE CLINICAL / REFERENCE FIELDS
# ================================================================

clinical_records = []

for pid in test_ids:

    patient_data = load_patient_data(pid)

    record = {
        "patient_id": pid
    }

    record.update(
        patient_data["clinical"]
    )

    clinical_records.append(record)


clinical_df = pd.DataFrame(
    clinical_records
)


print("=" * 70)
print("AVAILABLE INFO_4CH.CFG FIELDS")
print("=" * 70)

print("\nColumns:")
for column in clinical_df.columns:
    print(" -", column)


print("\nSample rows:")
display(
    clinical_df.head()
)


print("\nNon-null counts:")
display(
    clinical_df.notna().sum()
)

AVAILABLE INFO_4CH.CFG FIELDS

Columns:
 - patient_id
 - ED
 - ES
 - NbFrame
 - Sex
 - Age
 - ImageQuality
 - EF
 - FrameRate

Sample rows:


,patient_id,ED,ES,NbFrame,Sex,Age,ImageQuality,EF,FrameRate
0,patient0173,1,20,20,M,57,Good,59,56.5
1,patient0391,1,22,22,F,46,Good,35,55.4
2,patient0111,1,17,17,M,53,Medium,43,56.5
3,patient0080,1,20,20,M,72,Good,60,54.2
4,patient0465,1,23,23,F,85,Medium,45,55.4



Non-null counts:


,0
patient_id,50
ED,50
ES,50
NbFrame,50
Sex,50
Age,50
ImageQuality,50
EF,50
FrameRate,50


In [ ]:
# ================================================================
# PHASE 2A — CELL 14
# PHYSICAL LV GEOMETRY FUNCTIONS
# ================================================================

def mask_area_mm2(
    mask,
    spacing
):

    mask = np.asarray(
        mask
    ).astype(bool)

    spacing_x, spacing_y = map(
        float,
        spacing
    )

    return float(
        mask.sum()
        *
        spacing_x
        *
        spacing_y
    )


def mask_points_mm(
    mask,
    spacing
):

    mask = np.asarray(
        mask
    ).astype(bool)

    rows, cols = np.where(
        mask
    )

    spacing_x, spacing_y = map(
        float,
        spacing
    )

    # x corresponds to image columns
    # y corresponds to image rows

    x_mm = (
        cols.astype(np.float64)
        *
        spacing_x
    )

    y_mm = (
        rows.astype(np.float64)
        *
        spacing_y
    )

    return np.column_stack([
        x_mm,
        y_mm
    ])


def maximum_feret_mm(
    mask,
    spacing
):

    points = mask_points_mm(
        mask,
        spacing
    )

    if len(points) < 2:
        return np.nan


    hull = cv2.convexHull(
        points.astype(
            np.float32
        )
    ).reshape(-1, 2)


    if len(hull) < 2:
        return np.nan


    differences = (
        hull[:, None, :]
        -
        hull[None, :, :]
    )

    distances_squared = (
        differences ** 2
    ).sum(axis=2)


    return float(
        np.sqrt(
            distances_squared.max()
        )
    )


def pca_major_axis_mm(
    mask,
    spacing
):

    points = mask_points_mm(
        mask,
        spacing
    )

    if len(points) < 2:
        return np.nan


    centered = (
        points
        -
        points.mean(
            axis=0
        )
    )


    covariance = np.cov(
        centered,
        rowvar=False
    )


    eigenvalues, eigenvectors = (
        np.linalg.eigh(
            covariance
        )
    )


    principal_axis = (
        eigenvectors[
            :,
            np.argmax(
                eigenvalues
            )
        ]
    )


    projections = (
        centered
        @
        principal_axis
    )


    return float(
        projections.max()
        -
        projections.min()
    )


def area_length_volume_ml(
    area_mm2,
    length_mm
):

    if (
        not np.isfinite(area_mm2)
        or not np.isfinite(length_mm)
        or area_mm2 <= 0
        or length_mm <= 0
    ):
        return np.nan


    # Convert:
    # mm² -> cm²
    # mm  -> cm

    area_cm2 = (
        area_mm2 /
        100.0
    )

    length_cm = (
        length_mm /
        10.0
    )


    volume_ml = (
        8.0
        /
        (
            3.0
            *
            np.pi
        )
        *
        (
            area_cm2 ** 2
        )
        /
        length_cm
    )


    return float(
        volume_ml
    )


print(
    "✓ Physical LV geometry "
    "functions ready."
)

✓ Physical LV geometry functions ready.


In [ ]:
# ================================================================
# PHASE 2A — CELL 15
# SAMPLE GT GEOMETRY CHECK
# ================================================================

sample_geometry = []


for phase in [
    "ED",
    "ES"
]:

    phase_data = (
        sample_patient[
            phase
        ]
    )

    native_mask = (
        phase_data[
            "mask"
        ]
    )

    native_spacing = (
        phase_data[
            "spacing"
        ]
    )


    area_mm2 = mask_area_mm2(
        native_mask,
        native_spacing
    )


    feret_length = (
        maximum_feret_mm(
            native_mask,
            native_spacing
        )
    )


    pca_length = (
        pca_major_axis_mm(
            native_mask,
            native_spacing
        )
    )


    feret_volume = (
        area_length_volume_ml(
            area_mm2,
            feret_length
        )
    )


    pca_volume = (
        area_length_volume_ml(
            area_mm2,
            pca_length
        )
    )


    sample_geometry.append({

        "phase":
            phase,

        "area_mm2":
            area_mm2,

        "feret_length_mm":
            feret_length,

        "pca_length_mm":
            pca_length,

        "feret_volume_ml":
            feret_volume,

        "pca_volume_ml":
            pca_volume
    })


sample_geometry_df = pd.DataFrame(
    sample_geometry
)


display(
    sample_geometry_df.round(3)
)

,phase,area_mm2,feret_length_mm,pca_length_mm,feret_volume_ml,pca_volume_ml
0,ED,2859.580,78.917,77.592,87.954,89.455
1,ES,1616.388,68.290,66.432,32.475,33.383


In [ ]:
# ================================================================
# PHASE 2A — CELL 16
# GT GEOMETRY CANDIDATES — VALIDATION PATIENTS ONLY
# ================================================================

gt_geometry_records = []


for index, pid in enumerate(val_ids, start=1):

    patient_data = load_patient_data(pid)

    patient_record = {
        "patient_id": pid,
        "reference_EF_pct": float(
            patient_data["clinical"]["EF"]
        ),
        "image_quality": patient_data["clinical"]["ImageQuality"],
        "age": patient_data["clinical"]["Age"],
        "sex": patient_data["clinical"]["Sex"]
    }


    for phase in ["ED", "ES"]:

        mask = patient_data[phase]["mask"]
        spacing = patient_data[phase]["spacing"]

        area = mask_area_mm2(
            mask,
            spacing
        )

        feret_length = maximum_feret_mm(
            mask,
            spacing
        )

        pca_length = pca_major_axis_mm(
            mask,
            spacing
        )

        feret_volume = area_length_volume_ml(
            area,
            feret_length
        )

        pca_volume = area_length_volume_ml(
            area,
            pca_length
        )


        patient_record[
            f"{phase}_area_mm2"
        ] = area

        patient_record[
            f"{phase}_feret_length_mm"
        ] = feret_length

        patient_record[
            f"{phase}_pca_length_mm"
        ] = pca_length

        patient_record[
            f"{phase}_feret_volume_ml"
        ] = feret_volume

        patient_record[
            f"{phase}_pca_volume_ml"
        ] = pca_volume


    # -------------------------
    # Maximum-Feret quantities
    # -------------------------

    patient_record["feret_SV_ml"] = (
        patient_record["ED_feret_volume_ml"]
        -
        patient_record["ES_feret_volume_ml"]
    )

    patient_record["feret_EF_pct"] = (
        patient_record["feret_SV_ml"]
        /
        patient_record["ED_feret_volume_ml"]
        *
        100.0
    )


    # -------------------------
    # PCA quantities
    # -------------------------

    patient_record["pca_SV_ml"] = (
        patient_record["ED_pca_volume_ml"]
        -
        patient_record["ES_pca_volume_ml"]
    )

    patient_record["pca_EF_pct"] = (
        patient_record["pca_SV_ml"]
        /
        patient_record["ED_pca_volume_ml"]
        *
        100.0
    )


    gt_geometry_records.append(
        patient_record
    )


    if index % 10 == 0 or index == len(val_ids):

        print(
            f"Processed {index}/{len(val_ids)} "
            f"validation patients"
        )


gt_geometry_df = pd.DataFrame(
    gt_geometry_records
)


assert len(gt_geometry_df) == 50


GT_GEOMETRY_PATH = (
    P2A_TABLE_DIR /
    "validation_gt_geometry_candidates.csv"
)


gt_geometry_df.to_csv(
    GT_GEOMETRY_PATH,
    index=False
)


print("\n✓ Validation GT geometry complete.")

print("\nSaved:")
print(GT_GEOMETRY_PATH)

Processed 10/50 validation patients
Processed 20/50 validation patients
Processed 30/50 validation patients
Processed 40/50 validation patients
Processed 50/50 validation patients

✓ Validation GT geometry complete.

Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase2_quantitative/phase2A_measurement_method/tables/validation_gt_geometry_candidates.csv


In [ ]:
# ================================================================
# PHASE 2A — CELL 17
# GT GEOMETRY SANITY CHECKS
# ================================================================

geometry_checks = {

    "all_ED_area_positive":
        bool(
            (
                gt_geometry_df[
                    "ED_area_mm2"
                ] > 0
            ).all()
        ),

    "all_ES_area_positive":
        bool(
            (
                gt_geometry_df[
                    "ES_area_mm2"
                ] > 0
            ).all()
        ),

    "all_feret_lengths_positive":
        bool(
            (
                gt_geometry_df[
                    [
                        "ED_feret_length_mm",
                        "ES_feret_length_mm"
                    ]
                ] > 0
            ).all().all()
        ),

    "all_pca_lengths_positive":
        bool(
            (
                gt_geometry_df[
                    [
                        "ED_pca_length_mm",
                        "ES_pca_length_mm"
                    ]
                ] > 0
            ).all().all()
        ),

    "all_feret_EDV_positive":
        bool(
            (
                gt_geometry_df[
                    "ED_feret_volume_ml"
                ] > 0
            ).all()
        ),

    "all_pca_EDV_positive":
        bool(
            (
                gt_geometry_df[
                    "ED_pca_volume_ml"
                ] > 0
            ).all()
        ),

    "feret_EF_in_0_100":
        bool(
            gt_geometry_df[
                "feret_EF_pct"
            ].between(
                0,
                100
            ).all()
        ),

    "pca_EF_in_0_100":
        bool(
            gt_geometry_df[
                "pca_EF_pct"
            ].between(
                0,
                100
            ).all()
        )
}


for key, value in (
    geometry_checks.items()
):

    print(
        f"{key:30s}: "
        f"{'✓' if value else '✗'}"
    )


print("\n" + "=" * 70)
print("GT GEOMETRY SUMMARY")
print("=" * 70)


summary_columns = [

    "ED_area_mm2",
    "ES_area_mm2",

    "ED_feret_length_mm",
    "ES_feret_length_mm",

    "ED_pca_length_mm",
    "ES_pca_length_mm",

    "ED_feret_volume_ml",
    "ES_feret_volume_ml",

    "ED_pca_volume_ml",
    "ES_pca_volume_ml",

    "feret_EF_pct",
    "pca_EF_pct"
]


display(
    gt_geometry_df[
        summary_columns
    ]
    .describe()
    .T
    .round(3)
)

all_ED_area_positive          : ✓
all_ES_area_positive          : ✓
all_feret_lengths_positive    : ✓
all_pca_lengths_positive      : ✓
all_feret_EDV_positive        : ✓
all_pca_EDV_positive          : ✓
feret_EF_in_0_100             : ✓
pca_EF_in_0_100               : ✓

GT GEOMETRY SUMMARY


,count,mean,std,min,25%,50%,75%,max
ED_area_mm2,50.0,2859.929,651.285,1565.256,2404.281,2822.441,3309.117,4357.578
ES_area_mm2,50.0,1834.964,594.890,912.876,1483.910,1713.054,2071.806,3902.895
ED_feret_length_mm,50.0,80.076,8.491,64.126,74.029,80.217,86.776,99.591
ES_feret_length_mm,50.0,68.096,8.638,53.988,61.665,66.173,72.657,87.763
ED_pca_length_mm,50.0,78.760,8.664,61.728,72.514,78.527,85.070,98.772
ES_pca_length_mm,50.0,66.613,8.619,53.082,60.721,64.614,70.930,86.919
ED_feret_volume_ml,50.0,88.338,31.486,31.735,64.771,86.743,106.061,161.841
ES_feret_volume_ml,50.0,43.783,24.961,12.422,30.239,38.008,51.052,147.327
ED_pca_volume_ml,50.0,89.817,31.875,32.281,66.082,88.880,106.570,163.183
ES_pca_volume_ml,50.0,44.755,25.281,12.608,30.945,39.036,52.315,148.757


In [ ]:
# ================================================================
# PHASE 2A — CELL 18
# GEOMETRY METHOD COMPARISON AGAINST REFERENCE EF
# ================================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)


reference_ef = (
    gt_geometry_df[
        "reference_EF_pct"
    ].to_numpy(dtype=float)
)


comparison_records = []


for method in [
    "feret",
    "pca"
]:

    estimated_ef = (
        gt_geometry_df[
            f"{method}_EF_pct"
        ].to_numpy(dtype=float)
    )


    errors = (
        estimated_ef
        -
        reference_ef
    )


    mae = mean_absolute_error(
        reference_ef,
        estimated_ef
    )

    rmse = np.sqrt(
        mean_squared_error(
            reference_ef,
            estimated_ef
        )
    )

    bias = float(
        np.mean(errors)
    )

    correlation = float(
        np.corrcoef(
            reference_ef,
            estimated_ef
        )[0, 1]
    )


    comparison_records.append({

        "method":
            method,

        "EF_MAE_pp":
            mae,

        "EF_RMSE_pp":
            rmse,

        "EF_bias_pp":
            bias,

        "EF_correlation":
            correlation,

        "mean_estimated_EF":
            float(
                np.mean(
                    estimated_ef
                )
            ),

        "mean_reference_EF":
            float(
                np.mean(
                    reference_ef
                )
            )
    })


geometry_comparison_df = pd.DataFrame(
    comparison_records
)


display(
    geometry_comparison_df.round(4)
)


GEOMETRY_COMPARISON_PATH = (
    P2A_TABLE_DIR /
    "validation_geometry_method_comparison.csv"
)


geometry_comparison_df.to_csv(
    GEOMETRY_COMPARISON_PATH,
    index=False
)


print("\nSaved:")
print(GEOMETRY_COMPARISON_PATH)

,method,EF_MAE_pp,EF_RMSE_pp,EF_bias_pp,EF_correlation,mean_estimated_EF,mean_reference_EF
0,feret,8.2922,9.5364,6.0454,0.8512,51.5854,45.54
1,pca,8.1604,9.4344,5.8053,0.8475,51.3453,45.54



Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase2_quantitative/phase2A_measurement_method/tables/validation_geometry_method_comparison.csv


In [ ]:
# ================================================================
# PHASE 2A — CELL 19
# PATIENT-LEVEL FERET VS PCA COMPARISON
# ================================================================

paired_geometry_df = (
    gt_geometry_df[
        [
            "patient_id",
            "reference_EF_pct",
            "feret_EF_pct",
            "pca_EF_pct"
        ]
    ]
    .copy()
)


paired_geometry_df[
    "feret_abs_error_pp"
] = np.abs(

    paired_geometry_df[
        "feret_EF_pct"
    ]

    -

    paired_geometry_df[
        "reference_EF_pct"
    ]
)


paired_geometry_df[
    "pca_abs_error_pp"
] = np.abs(

    paired_geometry_df[
        "pca_EF_pct"
    ]

    -

    paired_geometry_df[
        "reference_EF_pct"
    ]
)


paired_geometry_df[
    "pca_minus_feret_abs_error_pp"
] = (

    paired_geometry_df[
        "pca_abs_error_pp"
    ]

    -

    paired_geometry_df[
        "feret_abs_error_pp"
    ]
)


feret_better = int(
    (
        paired_geometry_df[
            "feret_abs_error_pp"
        ]
        <
        paired_geometry_df[
            "pca_abs_error_pp"
        ]
    ).sum()
)


pca_better = int(
    (
        paired_geometry_df[
            "pca_abs_error_pp"
        ]
        <
        paired_geometry_df[
            "feret_abs_error_pp"
        ]
    ).sum()
)


ties = int(
    len(paired_geometry_df)
    -
    feret_better
    -
    pca_better
)


print("=" * 70)
print("PAIRED METHOD COMPARISON")
print("=" * 70)

print(
    "\nFeret lower EF error:",
    feret_better,
    "/ 50"
)

print(
    "PCA lower EF error  :",
    pca_better,
    "/ 50"
)

print(
    "Ties                :",
    ties
)

print(
    "\nMean PCA − Feret absolute EF error:"
)

print(
    f"{paired_geometry_df['pca_minus_feret_abs_error_pp'].mean():+.4f} pp"
)


display(
    paired_geometry_df.head().round(3)
)


PAIRED_PATH = (
    P2A_TABLE_DIR /
    "validation_geometry_paired_comparison.csv"
)


paired_geometry_df.to_csv(
    PAIRED_PATH,
    index=False
)

PAIRED METHOD COMPARISON

Feret lower EF error: 19 / 50
PCA lower EF error  : 31 / 50
Ties                : 0

Mean PCA − Feret absolute EF error:
-0.1318 pp


,patient_id,reference_EF_pct,feret_EF_pct,pca_EF_pct,feret_abs_error_pp,pca_abs_error_pp,pca_minus_feret_abs_error_pp
0,patient0037,49.0,53.997,53.294,4.997,4.294,-0.703
1,patient0332,48.0,53.870,53.352,5.870,5.352,-0.518
2,patient0350,36.0,50.786,50.659,14.786,14.659,-0.126
3,patient0471,48.0,58.606,57.942,10.606,9.942,-0.664
4,patient0137,53.0,60.856,60.942,7.856,7.942,0.086


In [ ]:
# ================================================================
# PHASE 2A — CELL 20
# FREEZE LONG-AXIS / GEOMETRY METHOD
# ================================================================

P2A_DECISION_PATH = (
    P2A_CONFIG_DIR /
    "phase2A_geometry_method_decision.json"
)


geometry_decision = {

    "status":
        "FROZEN",

    "selection_partition":
        "validation",

    "selection_n_patients":
        50,

    "selected_long_axis_method":
        "PCA_major_axis",

    "alternative_method":
        "maximum_feret",

    "selected_volume_method":
        "single-plane area-length",

    "volume_equation":
        "V = (8 / (3*pi)) * (A^2 / L)",

    "selection_reference":
        "CAMUS Info_4CH.cfg EF",

    "validation_results": {

        "PCA": {
            "EF_MAE_pp": 8.1604,
            "EF_RMSE_pp": 9.4344,
            "EF_bias_pp": 5.8053,
            "EF_correlation": 0.8475,
            "patients_lower_absolute_error": 31
        },

        "Maximum_Feret": {
            "EF_MAE_pp": 8.2922,
            "EF_RMSE_pp": 9.5364,
            "EF_bias_pp": 6.0454,
            "EF_correlation": 0.8512,
            "patients_lower_absolute_error": 19
        }
    },

    "mean_pca_minus_feret_absolute_EF_error_pp":
        -0.1318,

    "decision":
        (
            "PCA major-axis length selected because it produced "
            "slightly lower validation EF MAE, RMSE and bias, and "
            "lower patient-level absolute EF error in 31 of 50 "
            "validation patients."
        ),

    "interpretation_limitation":
        (
            "The difference between PCA and Maximum Feret was small. "
            "Selection applies only to the two tested long-axis "
            "definitions under this single-plane A4C area-length "
            "measurement protocol."
        ),

    "important_observation":
        (
            "Even with ground-truth masks, PCA-derived single-plane "
            "EF showed positive disagreement relative to CAMUS "
            "reference EF. Therefore downstream EF disagreement "
            "cannot be attributed solely to segmentation error."
        ),

    "test_data_used_for_method_selection":
        False,

    "method_changes_after_test_evaluation_allowed":
        False
}


with open(
    P2A_DECISION_PATH,
    "w"
) as f:

    json.dump(
        geometry_decision,
        f,
        indent=2
    )


print("=" * 70)
print("PHASE 2A GEOMETRY METHOD FROZEN")
print("=" * 70)

print("\nSelected long-axis method : PCA major axis")
print("Volume method             : Single-plane area-length")
print("Selection partition       : Validation only")
print("Patients                  : 50")

print("\nValidation EF:")
print("PCA MAE                   : 8.1604 pp")
print("Maximum Feret MAE         : 8.2922 pp")
print("PCA lower error patients  : 31 / 50")
print("Feret lower error patients: 19 / 50")

print(
    "\nIMPORTANT: PCA is selected among the two tested methods; "
    "the difference was small."
)

print("\n✓ Geometry method is now frozen.")
print("✓ Test data did not determine this decision.")

PHASE 2A GEOMETRY METHOD FROZEN

Selected long-axis method : PCA major axis
Volume method             : Single-plane area-length
Selection partition       : Validation only
Patients                  : 50

Validation EF:
PCA MAE                   : 8.1604 pp
Maximum Feret MAE         : 8.2922 pp
PCA lower error patients  : 31 / 50
Feret lower error patients: 19 / 50

IMPORTANT: PCA is selected among the two tested methods; the difference was small.

✓ Geometry method is now frozen.
✓ Test data did not determine this decision.


In [ ]:
# ================================================================
# PHASE 2A — CELL 21
# FROZEN SEGMENTATION POSTPROCESSING
# ================================================================

def keep_largest_component(mask):

    mask_uint8 = (
        np.asarray(mask)
        .astype(np.uint8)
    )

    if mask_uint8.sum() == 0:

        return np.zeros_like(
            mask_uint8,
            dtype=bool
        )


    n_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            mask_uint8,
            connectivity=8
        )
    )


    if n_labels <= 1:

        return np.zeros_like(
            mask_uint8,
            dtype=bool
        )


    largest_label = (
        1
        +
        np.argmax(
            stats[
                1:,
                cv2.CC_STAT_AREA
            ]
        )
    )


    return (
        labels
        ==
        largest_label
    )


def postprocess_prediction(
    probability_mask,
    threshold=0.50
):

    binary_mask = (
        probability_mask
        >=
        threshold
    )

    largest = (
        keep_largest_component(
            binary_mask
        )
    )

    filled = binary_fill_holes(
        largest
    )

    return np.asarray(
        filled
    ).astype(bool)


print(
    "✓ Frozen Phase 1 postprocessing ready."
)

✓ Frozen Phase 1 postprocessing ready.


In [ ]:
# ================================================================
# PHASE 2A — CELL 22
# LOAD FROZEN PHASE 1F MODEL
# ================================================================

tf.keras.backend.clear_session()
gc.collect()


frozen_segmentation_model = (
    tf.keras.models.load_model(
        FINAL_MODEL_PATH,
        compile=False
    )
)


assert (
    frozen_segmentation_model.count_params()
    ==
    7_759_521
)

assert (
    frozen_segmentation_model.input_shape
    ==
    (None, 256, 256, 1)
)


print("=" * 70)
print("FROZEN SEGMENTATION MODEL LOADED")
print("=" * 70)

print(
    "\nParameters:",
    f"{frozen_segmentation_model.count_params():,}"
)

print(
    "Input:",
    frozen_segmentation_model.input_shape
)

print(
    "\n✓ Inference model ready."
)

FROZEN SEGMENTATION MODEL LOADED

Parameters: 7,759,521
Input: (None, 256, 256, 1)

✓ Inference model ready.


In [ ]:
# ================================================================
# PHASE 2A — CELL 23
# GT VS PREDICTED QUANTITATIVE MEASUREMENTS — VALIDATION
# ================================================================

validation_measurement_records = []


for index, pid in enumerate(
    val_ids,
    start=1
):

    patient_data = (
        load_patient_data(pid)
    )


    record = {

        "patient_id":
            pid,

        "reference_EF_pct":
            float(
                patient_data[
                    "clinical"
                ][
                    "EF"
                ]
            ),

        "image_quality":
            patient_data[
                "clinical"
            ][
                "ImageQuality"
            ]
    }


    for phase in ["ED", "ES"]:

        phase_data = (
            patient_data[phase]
        )


        image = (
            phase_data["image"]
        )

        gt_native_mask = (
            phase_data["mask"]
        )

        native_spacing = (
            phase_data["spacing"]
        )


        # --------------------------------
        # Frozen Phase 1 preprocessing
        # --------------------------------

        processed_image, scale_factors = (
            preprocess_for_segmentation(
                image,
                target_size=256
            )
        )


        resized_gt_mask = (
            preprocess_mask(
                gt_native_mask,
                target_size=256
            )
        )


        resized_spacing = (
            effective_spacing(
                native_spacing,
                scale_factors
            )
        )


        # --------------------------------
        # Frozen model inference
        # --------------------------------

        model_input = (
            processed_image[
                None,
                ...,
                None
            ]
        )


        probability = (
            frozen_segmentation_model(
                model_input,
                training=False
            )
            .numpy()[0, :, :, 0]
        )


        predicted_mask = (
            postprocess_prediction(
                probability,
                threshold=0.50
            )
        )


        # --------------------------------
        # GT quantitative geometry
        # --------------------------------

        gt_area = mask_area_mm2(
            resized_gt_mask,
            resized_spacing
        )

        gt_length = pca_major_axis_mm(
            resized_gt_mask,
            resized_spacing
        )

        gt_volume = area_length_volume_ml(
            gt_area,
            gt_length
        )


        # --------------------------------
        # Predicted quantitative geometry
        # --------------------------------

        pred_area = mask_area_mm2(
            predicted_mask,
            resized_spacing
        )

        pred_length = pca_major_axis_mm(
            predicted_mask,
            resized_spacing
        )

        pred_volume = area_length_volume_ml(
            pred_area,
            pred_length
        )


        record[
            f"{phase}_GT_area_mm2"
        ] = gt_area

        record[
            f"{phase}_PRED_area_mm2"
        ] = pred_area

        record[
            f"{phase}_GT_length_mm"
        ] = gt_length

        record[
            f"{phase}_PRED_length_mm"
        ] = pred_length

        record[
            f"{phase}_GT_volume_ml"
        ] = gt_volume

        record[
            f"{phase}_PRED_volume_ml"
        ] = pred_volume


    # ------------------------------------------------------------
    # GT-derived patient quantities
    # ------------------------------------------------------------

    record["GT_SV_ml"] = (
        record["ED_GT_volume_ml"]
        -
        record["ES_GT_volume_ml"]
    )


    record["GT_EF_pct"] = (
        record["GT_SV_ml"]
        /
        record["ED_GT_volume_ml"]
        *
        100.0
    )


    # ------------------------------------------------------------
    # Prediction-derived patient quantities
    # ------------------------------------------------------------

    record["PRED_SV_ml"] = (
        record["ED_PRED_volume_ml"]
        -
        record["ES_PRED_volume_ml"]
    )


    record["PRED_EF_pct"] = (
        record["PRED_SV_ml"]
        /
        record["ED_PRED_volume_ml"]
        *
        100.0
    )


    validation_measurement_records.append(
        record
    )


    if (
        index % 10 == 0
        or index == len(val_ids)
    ):

        print(
            f"Processed "
            f"{index}/{len(val_ids)} "
            f"validation patients"
        )


validation_measurements_df = pd.DataFrame(
    validation_measurement_records
)


assert (
    len(
        validation_measurements_df
    )
    ==
    50
)


VALIDATION_MEASUREMENTS_PATH = (
    P2A_TABLE_DIR /
    "validation_gt_vs_pred_quantitative_measurements.csv"
)


validation_measurements_df.to_csv(
    VALIDATION_MEASUREMENTS_PATH,
    index=False
)


print(
    "\n✓ Validation quantitative "
    "measurements complete."
)

print("\nSaved:")
print(
    VALIDATION_MEASUREMENTS_PATH
)

Processed 10/50 validation patients
Processed 20/50 validation patients
Processed 30/50 validation patients
Processed 40/50 validation patients
Processed 50/50 validation patients

✓ Validation quantitative measurements complete.

Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase2_quantitative/phase2A_measurement_method/tables/validation_gt_vs_pred_quantitative_measurements.csv


In [ ]:
# ================================================================
# PHASE 2A — CELL 24
# SEGMENTATION → MEASUREMENT ERROR
# ================================================================

def error_summary(
    reference,
    estimate
):

    reference = np.asarray(
        reference,
        dtype=float
    )

    estimate = np.asarray(
        estimate,
        dtype=float
    )


    error = (
        estimate
        -
        reference
    )


    absolute_error = (
        np.abs(error)
    )


    mae = float(
        np.mean(
            absolute_error
        )
    )

    rmse = float(
        np.sqrt(
            np.mean(
                error ** 2
            )
        )
    )

    bias = float(
        np.mean(
            error
        )
    )

    correlation = float(
        np.corrcoef(
            reference,
            estimate
        )[0, 1]
    )


    return {
        "MAE": mae,
        "RMSE": rmse,
        "bias": bias,
        "correlation": correlation
    }


measurement_error_records = []


measurement_pairs = [

    (
        "ED_area_mm2",
        "ED_GT_area_mm2",
        "ED_PRED_area_mm2"
    ),

    (
        "ES_area_mm2",
        "ES_GT_area_mm2",
        "ES_PRED_area_mm2"
    ),

    (
        "ED_length_mm",
        "ED_GT_length_mm",
        "ED_PRED_length_mm"
    ),

    (
        "ES_length_mm",
        "ES_GT_length_mm",
        "ES_PRED_length_mm"
    ),

    (
        "EDV_ml",
        "ED_GT_volume_ml",
        "ED_PRED_volume_ml"
    ),

    (
        "ESV_ml",
        "ES_GT_volume_ml",
        "ES_PRED_volume_ml"
    ),

    (
        "SV_ml",
        "GT_SV_ml",
        "PRED_SV_ml"
    ),

    (
        "EF_percentage_points",
        "GT_EF_pct",
        "PRED_EF_pct"
    )
]


for (
    quantity,
    gt_column,
    pred_column
) in measurement_pairs:

    stats = error_summary(

        validation_measurements_df[
            gt_column
        ],

        validation_measurements_df[
            pred_column
        ]
    )


    measurement_error_records.append({

        "quantity":
            quantity,

        **stats
    })


measurement_error_df = pd.DataFrame(
    measurement_error_records
)


display(
    measurement_error_df.round(4)
)


MEASUREMENT_ERROR_PATH = (
    P2A_TABLE_DIR /
    "validation_segmentation_to_measurement_error.csv"
)


measurement_error_df.to_csv(
    MEASUREMENT_ERROR_PATH,
    index=False
)

,quantity,MAE,RMSE,bias,correlation
0,ED_area_mm2,164.5318,226.0635,-54.6331,0.9453
1,ES_area_mm2,153.7326,201.3945,-12.7039,0.9464
2,ED_length_mm,2.7711,3.4020,0.0329,0.9183
3,ES_length_mm,2.5043,3.6552,-0.1635,0.9031
4,EDV_ml,8.7770,12.4116,-3.9520,0.9378
5,ESV_ml,6.4650,8.4372,-1.1048,0.9544
6,SV_ml,8.0431,9.8317,-2.8472,0.8719
7,EF_percentage_points,7.0468,9.5155,-2.1015,0.7740


In [ ]:
# ================================================================
# PHASE 2A — CELL 25
# EF ERROR DECOMPOSITION
# ================================================================

reference_ef = (
    validation_measurements_df[
        "reference_EF_pct"
    ].to_numpy(dtype=float)
)

gt_geometry_ef = (
    validation_measurements_df[
        "GT_EF_pct"
    ].to_numpy(dtype=float)
)

predicted_ef = (
    validation_measurements_df[
        "PRED_EF_pct"
    ].to_numpy(dtype=float)
)


ef_decomposition = []


# ------------------------------------------------
# 1. Reference EF -> GT-mask geometry EF
# ------------------------------------------------

stats = error_summary(
    reference_ef,
    gt_geometry_ef
)

ef_decomposition.append({

    "comparison":
        "Reference EF → GT-mask geometry EF",

    "interpretation":
        "Geometry/reference disagreement",

    **stats
})


# ------------------------------------------------
# 2. GT-mask geometry EF -> predicted-mask EF
# ------------------------------------------------

stats = error_summary(
    gt_geometry_ef,
    predicted_ef
)

ef_decomposition.append({

    "comparison":
        "GT-mask geometry EF → predicted-mask EF",

    "interpretation":
        "Segmentation-induced downstream disagreement",

    **stats
})


# ------------------------------------------------
# 3. Reference EF -> final predicted-mask EF
# ------------------------------------------------

stats = error_summary(
    reference_ef,
    predicted_ef
)

ef_decomposition.append({

    "comparison":
        "Reference EF → predicted-mask EF",

    "interpretation":
        "Total end-to-end disagreement",

    **stats
})


ef_decomposition_df = pd.DataFrame(
    ef_decomposition
)


display(
    ef_decomposition_df.round(4)
)


EF_DECOMPOSITION_PATH = (
    P2A_TABLE_DIR /
    "validation_EF_error_decomposition.csv"
)


ef_decomposition_df.to_csv(
    EF_DECOMPOSITION_PATH,
    index=False
)


print("\nSaved:")
print(EF_DECOMPOSITION_PATH)

,comparison,interpretation,MAE,RMSE,bias,correlation
0,Reference EF → GT-mask geometry EF,Geometry/reference disagreement,8.1076,9.3858,5.7688,0.8488
1,GT-mask geometry EF → predicted-mask EF,Segmentation-induced downstream disagreement,7.0468,9.5155,-2.1015,0.7740
2,Reference EF → predicted-mask EF,Total end-to-end disagreement,8.7448,11.1300,3.6672,0.6944



Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase2_quantitative/phase2A_measurement_method/tables/validation_EF_error_decomposition.csv


In [ ]:
# ================================================================
# PHASE 2A — CELL 26
# FREEZE DEVELOPMENT-STAGE QUANTITATIVE PIPELINE
# ================================================================

PHASE2A_FINAL_RECORD_PATH = (
    P2A_CONFIG_DIR /
    "phase2A_development_record.json"
)


phase2a_development_record = {

    "status":
        "DEVELOPMENT_COMPLETE_AND_FROZEN",

    "selection_partition":
        "validation",

    "selection_n_patients":
        50,

    "segmentation_model": {
        "status":
            "FROZEN_FROM_PHASE1F",

        "path":
            str(FINAL_MODEL_PATH),

        "sha256":
            EXPECTED_MODEL_SHA256,

        "threshold":
            0.50,

        "postprocessing": [
            "largest connected component",
            "binary hole filling"
        ]
    },

    "measurement_pipeline": {

        "resize":
            "256x256 stretch resize",

        "physical_spacing":
            (
                "Original NIfTI spacing adjusted independently "
                "for x and y according to resize scale factors."
            ),

        "area":
            "binary LV cavity area in mm^2",

        "long_axis":
            "PCA major axis",

        "volume":
            "single-plane area-length",

        "volume_equation":
            "V = (8/(3*pi)) * A^2/L",

        "EDV":
            "ED single-plane area-length volume",

        "ESV":
            "ES single-plane area-length volume",

        "SV":
            "EDV - ESV",

        "EF":
            "(EDV - ESV)/EDV * 100"
    },

    "geometry_method_selection": {

        "candidates": [
            "Maximum Feret",
            "PCA major axis"
        ],

        "selected":
            "PCA major axis",

        "PCA_validation_EF_MAE_pp":
            8.1604,

        "Feret_validation_EF_MAE_pp":
            8.2922,

        "PCA_lower_error_patients":
            31,

        "Feret_lower_error_patients":
            19,

        "note":
            (
                "PCA showed only a small advantage. "
                "The selection applies to this predefined "
                "two-method comparison."
            )
    },

    "validation_error_decomposition": {

        "reference_to_GT_geometry": {
            "EF_MAE_pp": 8.1076,
            "EF_RMSE_pp": 9.3858,
            "EF_bias_pp": 5.7688,
            "correlation": 0.8488
        },

        "GT_geometry_to_predicted_geometry": {
            "EF_MAE_pp": 7.0468,
            "EF_RMSE_pp": 9.5155,
            "EF_bias_pp": -2.1015,
            "correlation": 0.7740
        },

        "reference_to_predicted_geometry": {
            "EF_MAE_pp": 8.7448,
            "EF_RMSE_pp": 11.1300,
            "EF_bias_pp": 3.6672,
            "correlation": 0.6944
        }
    },

    "interpretation": {

        "geometry_component":
            (
                "Substantial EF disagreement remains even when "
                "ground-truth LV masks are used, showing that "
                "single-plane geometric estimation/reference "
                "differences contribute independently of "
                "segmentation."
            ),

        "segmentation_component":
            (
                "Predicted masks introduce additional error into "
                "area, length, volume and EF measurements."
            ),

        "error_additivity":
            (
                "Error components must not be treated as directly "
                "additive because signed errors can interact and "
                "partially cancel."
            )
    },

    "frozen_before_test_quantitative_evaluation":
        True,

    "test_results_may_change_pipeline":
        False
}


with open(
    PHASE2A_FINAL_RECORD_PATH,
    "w"
) as f:

    json.dump(
        phase2a_development_record,
        f,
        indent=2
    )


print("=" * 70)
print("PHASE 2A DEVELOPMENT PIPELINE FROZEN")
print("=" * 70)

print("\nLong-axis        : PCA major axis")
print("Volume           : Single-plane area-length")
print("Segmentation     : Frozen Phase 1F model")
print("Threshold        : 0.50")
print("Postprocessing   : LCC + hole filling")
print("Selection data   : Validation only")

print("\nNo further method selection or tuning is allowed")
print("from the quantitative test results.")

print("\n✓ Ready for locked quantitative evaluation.")

PHASE 2A DEVELOPMENT PIPELINE FROZEN

Long-axis        : PCA major axis
Volume           : Single-plane area-length
Segmentation     : Frozen Phase 1F model
Threshold        : 0.50
Postprocessing   : LCC + hole filling
Selection data   : Validation only

No further method selection or tuning is allowed
from the quantitative test results.

✓ Ready for locked quantitative evaluation.
